In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py
import numpy as np
np.set_printoptions(suppress=True)


## merge single track to annual (caution for 2006,2007,2017, see notes)

In [125]:
import xarray as xr
p="/data/ggong/CALIPSO/Dustprof/ENA_5x5/Fine-Mode_Coarse-Mode_Pure-Dust-V1-CAL_LID_L2_05km-V4-21.2021*.nc"
ds = xr.open_mfdataset(
    p, combine="nested", concat_dim="time",
    preprocess=lambda d: d.swap_dims({"CAL_L2_5km_Pro":"time"}).rename({"Alt":"height"}).sortby("time")
)


In [126]:
ds 

<xarray.Dataset> Size: 198MB
Dimensions:                                       (time: 15492, height: 399)
Coordinates:
  * time                                          (time) datetime64[ns] 124kB ...
    lat                                           (time) float32 62kB dask.array<chunksize=(20,), meta=np.ndarray>
    lon                                           (time) float32 62kB dask.array<chunksize=(20,), meta=np.ndarray>
    height                                        (height) float64 3kB 29.8 ....
Data variables:
    Day_Night_Flag                                (time) float32 62kB dask.array<chunksize=(20,), meta=np.ndarray>
    AVD_Aerosol_Subtype                           (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    AVD_Feature_Type                              (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Surface_Elevation                             (time) float32 62kB dask.array<chunksize=(20,), meta=np.ndarray>
    Pure_Dust_Fine_Backscatter_Coefficient_532    (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Pure_Dust_Coarse_Backscatter_Coefficient_532  (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Pure_Dust_Fine_Extinction_Coefficient_532     (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Pure_Dust_Coarse_Extinction_Coefficient_532   (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Pure_Dust_Fine_Mass_Concentration             (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>
    Pure_Dust_Coarse_Mass_Concentration           (time, height) float32 25MB dask.array<chunksize=(20, 399), meta=np.ndarray>

In [127]:
# apply only when "heihgt" have 1 dimentions

In [128]:
ds.to_netcdf("/data/ggong/CALIPSO/Dustprof/ENA_annual/2021_5x5.nc")

In [74]:
# apply only when "heihgt" have 2 dimentions

In [113]:
ds = ds.assign_coords(height=ds.height.isel(time=0).values)  # apply only when "heihgt" have 2 dimentions
ds = ds.drop_indexes("height") # follows above
ds.to_netcdf("/data/ggong/CALIPSO/Dustprof/ENA_annual/2017_5x5.nc")

## merge annual data

In [129]:
import xarray as xr

dssss = xr.open_mfdataset(
    "/data/ggong/CALIPSO/Dustprof/ENA_annual/*5x5.nc",
    combine="nested", concat_dim="time",
    preprocess=lambda d: d.sortby("height"),
    coords="minimal", compat="override", join="override"
).sortby("time")

dssss = dssss.assign_coords(height=dssss.height.values[::-1] if dssss.height.values[0] > dssss.height.values[-1] else dssss.height.values)

dssss.to_netcdf("/data/ggong/CALIPSO/Dustprof/ENA_annual/merge_06-21_5x5.nc")